# Đo throughput runner XE với cache thật
Dùng GPU T4 ×2. Gắn visual-cache, mscoco-2014, csa-semantic-banks-v130 và csa-prompt-full-karpathy-v175. Gửi csa_full_runner_throughput_review.zip.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile
from pathlib import Path
REVISION='423fac4e244ce125abe4f30b99353a72852e4087'
MAPPING={'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/full_runtime.py': 'full_runtime.py', 'experiments/clip_patch_baseline/run_full.py': 'run_full.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/score_full.py': 'score_full.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/test_full_data.py': 'test_full_data.py', 'experiments/clip_patch_baseline/make_full_notebook.py': 'make_full_notebook.py', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'FULL_XE10_PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/clip_patch_baseline/local_stage.py': 'local_stage.py', 'experiments/clip_patch_baseline/test_local_stage.py': 'test_local_stage.py', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': 'LOCAL_STAGE_V3_PROTOCOL.md', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/semantic_training.py': 'semantic_training.py', 'experiments/clip_patch_baseline/test_semantic_data.py': 'test_semantic_data.py', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': 'BANK_PROMPT_SCRATCH_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'bank_words_corrected.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/clip_patch_baseline/memory_fusion.py': 'memory_fusion.py', 'experiments/clip_patch_baseline/csa_full_data.py': 'csa_full_data.py', 'experiments/clip_patch_baseline/csa_full_runtime.py': 'csa_full_runtime.py', 'experiments/clip_patch_baseline/run_csa_full.py': 'run_csa_full.py', 'experiments/clip_patch_baseline/score_csa_full.py': 'score_csa_full.py', 'experiments/clip_patch_baseline/test_csa_full_data.py': 'test_csa_full_data.py', 'experiments/clip_patch_baseline/csa_patch_model.py': 'csa_patch_model.py', 'experiments/clip_patch_baseline/csa_shared_model.py': 'csa_shared_model.py', 'experiments/clip_patch_baseline/csa_sharing_data.py': 'csa_sharing_data.py', 'experiments/clip_patch_baseline/gradient_agreement.py': 'gradient_agreement.py', 'experiments/clip_patch_baseline/test_gradient_agreement.py': 'test_gradient_agreement.py', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': 'CSA_SHARED_FULL_XE10_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': 'make_csa_full_notebook.py'}
HASHES={'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/full_runtime.py': '230f7e10d96005fbffc33fb151a7a1d7c7245326e73826134a400772dadd04bb', 'experiments/clip_patch_baseline/run_full.py': '32dfb92228fad22d4f503680db1c8226db1e6abf9d7dd253786c3f16be9a97c0', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/score_full.py': 'fed170a6d12d1ab1bb969ae7eac29ee0855059417f6e2e876da193e279ca819a', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/test_full_data.py': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431', 'experiments/clip_patch_baseline/make_full_notebook.py': '2417f0c691e291808b9f3b44f8bdb60090fcaf8bb487d71948538a7d61d5679d', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/clip_patch_baseline/local_stage.py': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b', 'experiments/clip_patch_baseline/test_local_stage.py': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/clip_patch_baseline/semantic_training.py': '001f05f3e6959570571135a55abb7bc740217a447371bab627af8c72e0f26d11', 'experiments/clip_patch_baseline/test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': '93d1ebef1d359d393323951a152f50d16559c0fd235dac27eda4940653399a0c', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/csa_full_data.py': '4028cd2b656d2244cabe2b4ab510b7a701bd8d462157adce62e10bf2060ca8c1', 'experiments/clip_patch_baseline/csa_full_runtime.py': '508b10b38871041a7f4964d73291a82f55b7dae42a3e6651b82e1a451fff1325', 'experiments/clip_patch_baseline/run_csa_full.py': '3c5639e29b8a5137c3e677ae110d87cb697288e93bdbc8826ccd009ffd4ff7bd', 'experiments/clip_patch_baseline/score_csa_full.py': '25d912e57a96be0ac2cdef102d681ebff2afb00521de4e9ba7b2eea208d8bcb8', 'experiments/clip_patch_baseline/test_csa_full_data.py': '4f3f5e84ec751a54fe2a338441c9c6644398cbac950b4001c015b0a959e63b07', 'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/csa_shared_model.py': '9599759a16e8570fbb625c88c821f2cf26776be8a317db650adcd34ef563cbf8', 'experiments/clip_patch_baseline/csa_sharing_data.py': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243', 'experiments/clip_patch_baseline/gradient_agreement.py': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c', 'experiments/clip_patch_baseline/test_gradient_agreement.py': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': 'c9f4dd7b6507218cc63cd6c72809d2cf97670ce0b1172c6bb5159ef6d25b604a', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': 'c9f8e3e793b0841df24e938103cbf8a83396a297ded1634108afec10e2cead9b'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
RESUME_ROOT='' # Optional preserved full-run output from THIS exact recipe. Blank starts fresh.
WALL_HOURS=0.75 # Parent stop cap, not ETA/billed quota. Do not auto-extend.
OUTPUT=Path('/kaggle/working/csa_full_runner_throughput')
for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh runtime/output path')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-full10-',dir='/tmp'));DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair'
cfg=dict(csa_shared_full=True,semantic=True,bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),feature_storage='local_fp16_memmap_v2',prepare_seconds=5400,output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600,resume_root=RESUME_ROOT,chair=str(CHAIR))
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned asset: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

ACCEL_DEPS=SOURCE/'accelerate-deps'
os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Tail27 probe active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
PROBE_SOURCE="import sys,time\nfrom datetime import timedelta\nfrom pathlib import Path\nfrom statistics import median\nimport numpy as np\nfrom real_bank_data import read,save\nfrom full_data import prepare,load_stage,FeatureReader,TRAIN_IMAGES\nfrom csa_full_data import order_for_epoch\nfrom csa_sharing_data import image_plan,local_events\nfrom semantic_data import checked_inputs,record_arrays,SemanticContext\nfrom gradient_agreement import gradient_agreement\n\ndef stage(cfg):\n    start=time.monotonic()\n    prepare(cfg)\n    meta=time.monotonic()-start\n    root=Path(cfg['stage'])\n    data,_,_=load_stage(root)\n    order=order_for_epoch(data,1)\n    events=order[:32*24]\n    tail=order[-27:]\n    validation=np.arange(TRAIN_IMAGES,TRAIN_IMAGES+64)\n    selected=np.unique(np.r_[data['caption_image'][np.r_[events,tail]],validation])\n    ids=[int(x) for x in data['ids'][selected]]\n    remap={int(v):i for i,v in enumerate(selected)}\n    local_images=np.asarray([remap[int(i)] for i in data['caption_image'][np.r_[events,tail]]],dtype=np.int64)\n    _,source,_,texts,vectors,_,banks,_=checked_inputs(cfg)\n    wanted=set(ids)\n    records={}\n    with (source/'prompt_links.jsonl').open(encoding='utf8') as stream:\n        for row in map(__import__('json').loads,stream):\n            cid=row['coco_id']\n            if cid in wanted:\n                if cid in records:\n                    raise ValueError('Duplicate selected prompt')\n                records[cid]=record_arrays(row,texts)\n    if set(records)!=wanted:\n        raise ValueError('Missing selected prompts')\n    for name in records[ids[0]]:\n        np.save(root/f'sem_{name}.npy',np.stack([records[i][name] for i in ids]))\n    np.save(root/'sem_text_vectors.npy',vectors)\n    for name,value in banks.items():\n        np.save(root/f'sem_{name}_bank.npy',value)\n    reader=FeatureReader(root)\n    fetch_start=time.monotonic()\n    try:\n        patches=np.stack([reader.get(i) for i in selected])\n    finally:\n        reader.close()\n    reads=time.monotonic()-fetch_start\n    np.savez(root/'probe_inputs.npz',patches=patches,sequences=data['sequences'][np.r_[events,tail]],\n             image=local_images,valid=np.asarray([remap[int(i)] for i in validation]))\n    save(Path(cfg['output'])/'stage_probe.json',dict(images=len(ids),metadata_seconds=meta,\n         selected_patch_read_seconds=reads,total_stage_seconds=time.monotonic()-start,\n         train_caption_events=len(events),tail_events=len(tail),valid_images=len(validation)))\n\ndef bench(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from accelerate import Accelerator\n    from accelerate.utils import DistributedDataParallelKwargs,InitProcessGroupKwargs\n    from csa_shared_model import SharedCSAPatchCaptioner\n    from patch_beam import beam_search,check_paths\n    root=Path(cfg['stage'])\n    out=Path(cfg['output'])\n    accelerator=Accelerator(mixed_precision='no',dynamo_backend='no',kwargs_handlers=[\n        DistributedDataParallelKwargs(broadcast_buffers=False),\n        InitProcessGroupKwargs(backend='nccl',timeout=timedelta(seconds=300))])\n    if accelerator.num_processes!=2 or accelerator.device.type!='cuda':\n        raise ValueError('Select T4 x2')\n    torch.set_num_threads(1)\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    rank=accelerator.process_index\n    device=accelerator.device\n    with np.load(root/'probe_inputs.npz',allow_pickle=False) as z:\n        data={k:z[k] for k in z.files}\n    context=SemanticContext(root)\n    words=read(root/'vocabulary.json')\n    banks=[torch.from_numpy(np.load(root/f'sem_{k}_bank.npy',allow_pickle=False)) for k in ('noun','verb')]\n    def build(train):\n        torch.manual_seed(42)\n        raw=SharedCSAPatchCaptioner(len(words),*banks).to(device)\n        raw.train(train)\n        return raw,accelerator.prepare_model(raw)\n    def step(model,events,shared):\n        local=local_events(events,rank)\n        first,inverse=image_plan(data['image'],local,shared)\n        frames=data['image'][local[first]]\n        x=torch.from_numpy(np.asarray(data['patches'][frames],np.float32)).to(device)\n        seq=data['sequences'][local].astype(np.int64)\n        seq=torch.from_numpy(np.ascontiguousarray(seq[:,:int(np.count_nonzero(seq,axis=1).max())])).to(device)\n        prompt=context.images(frames,device)\n        gather=None if inverse is None else torch.from_numpy(inverse).to(device)\n        scores=model(x,seq[:,:-1],prompt,gather)\n        nll=F.cross_entropy(scores.flatten(0,1),seq[:,1:].reshape(-1),ignore_index=0,reduction='sum')\n        if not torch.isfinite(nll):\n            raise ValueError('Nonfinite benchmark loss')\n        tokens=int(np.count_nonzero(data['sequences'][events,1:]))\n        accelerator.backward(nll*(2/tokens))\n        total=torch.tensor([nll.detach().item(),len(frames)],device=device,dtype=torch.float64)\n        dist.all_reduce(total)\n        return float(total[0]/tokens),int(total[1].item())\n\n    raw,model=build(False)\n    reports=[]\n    for label,events in [('first32',np.arange(32)),('tail27',np.arange(768,795))]:\n        losses=[]\n        grads=None\n        failed=[]\n        for shared in (False,True):\n            model.zero_grad(set_to_none=True)\n            loss,_=step(model,events,shared)\n            losses.append(loss)\n            if rank==0:\n                if grads is None:\n                    grads={k:v.grad.detach().cpu().numpy().copy() for k,v in raw.named_parameters()}\n                else:\n                    failed=[k for k,v in raw.named_parameters()\n                            if not gradient_agreement(v.grad.detach().cpu().numpy(),grads[k])['passed']]\n        if rank==0:\n            reports.append(dict(case=label,losses=losses,failed=failed,\n                                passed=not failed and abs(losses[0]-losses[1])<=5e-5))\n    model.zero_grad(set_to_none=True)\n    dist.barrier()\n    torch.cuda.synchronize(device)\n    started=time.monotonic()\n    with torch.inference_mode():\n        examples=data['valid'][rank::2]\n        for start in range(0,len(examples),8):\n            frames=examples[start:start+8]\n            x=torch.from_numpy(np.asarray(data['patches'][frames],np.float32)).to(device)\n            memory=raw.encode(x,context.images(frames,device))\n            seq,scores=beam_search(raw,memory,width=5,maximum=20)\n            check_paths(raw,memory,seq,scores)\n    torch.cuda.synchronize(device)\n    decode=torch.tensor(time.monotonic()-started,device=device,dtype=torch.float64)\n    dist.all_reduce(decode,op=dist.ReduceOp.MAX)\n    del raw,model\n    accelerator.free_memory()\n    passes=[]\n    for shared in (False,True,True,False):\n        raw,model=build(True)\n        torch.cuda.manual_seed(4242+rank)\n        optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)\n        timings=[]\n        images=[]\n        torch.cuda.reset_peak_memory_stats(device)\n        for i in range(24):\n            dist.barrier()\n            torch.cuda.synchronize(device)\n            started=time.monotonic()\n            optimizer.zero_grad(set_to_none=True)\n            _,n=step(model,np.arange(i*32,(i+1)*32),shared)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)\n            optimizer.step()\n            torch.cuda.synchronize(device)\n            elapsed=torch.tensor(time.monotonic()-started,device=device,dtype=torch.float64)\n            dist.all_reduce(elapsed,op=dist.ReduceOp.MAX)\n            if i>=8:\n                timings.append(float(elapsed))\n                images.append(n)\n        peak=torch.tensor(torch.cuda.max_memory_allocated(device),device=device,dtype=torch.int64)\n        dist.all_reduce(peak,op=dist.ReduceOp.MAX)\n        passes.append(dict(shared=shared,p50_seconds=median(timings),\n                           seconds=timings,encoder_rows=images,peak_bytes=int(peak)))\n        del optimizer,raw,model\n        accelerator.free_memory()\n        torch.cuda.empty_cache()\n    if rank==0:\n        save(out/'result.json',dict(complete=True,parity=reports,passes=passes,\n              valid_decode64_seconds=float(decode),device=torch.cuda.get_device_name(device),\n              full_train=False,quality_scored=False))\n    accelerator.end_training()\n\nif __name__=='__main__':\n    {'stage':stage,'bench':bench}[sys.argv[1]](read(Path(sys.argv[2])))\n"
complete=False
try:
    for name,destination in MAPPING.items():download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,HASHES[name],SOURCE/destination)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_local_stage.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_csa_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_gradient_agreement.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)

    run([sys.executable,'-m','pip','download','-q','--no-deps','--only-binary=:all:','--dest',SOURCE,'accelerate==1.15.0'],180)
    wheel=SOURCE/'accelerate-1.15.0-py3-none-any.whl'
    if hashlib.sha256(wheel.read_bytes()).hexdigest()!='97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec':raise ValueError('Changed Accelerate wheel')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',ACCEL_DEPS,wheel],180)
    run([sys.executable,'-c',"import accelerate;assert accelerate.__version__=='1.15.0';print('PinnedAccelerate',accelerate.__version__,flush=True)"],120)
    run([sys.executable,'-c',"from csa_shared_model import SharedCSAPatchCaptioner;from csa_full_runtime import make_model;print('Neural module imports PASS; no model instantiated',flush=True)"],120)
    (OUTPUT/'accelerate_install.json').write_text(json.dumps(dict(version='1.15.0',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),dependencies_reinstalled=False)))

    script=SOURCE/'probe_full_runner_throughput.py'
    script.write_text(PROBE_SOURCE,encoding='utf8')
    run([sys.executable,'-u',script,'stage',CONFIG],1500)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',script,'bench',CONFIG],900)
    complete=(OUTPUT/'result.json').is_file()
    if not complete:
        raise RuntimeError('Missing throughput report')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc())
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,full_train=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for path in OUTPUT.iterdir():
            if path.is_file():z.write(path,path.name)
    print('SEND REVIEW:',review,flush=True)
